# Подготовка ML датасета для Backblaze

Собираем новый датасет для задачи предсказания отказа диска в горизонте 7 дней.

Исправляем ключевые проблемы старой версии:
- не используем leakage из будущего при формировании признаков и target;
- разделяем выборку по `serial_number`, чтобы одни и те же диски не попадали одновременно в train и test.


## Pipeline

1. Читает ежедневные CSV по одному файлу и берет только нужные колонки.
2. Строит target: `label = 1`, если отказ случится в ближайшие 7 дней.
3. Удаляет строки дня отказа.
4. Строит признаки и разности только по прошлым наблюдениям.
5. Делит данные по дискам, а не по строкам.
6. Делает downsampling только в train.
7. Сохраняет итог в `parquet`.


In [ ]:
from pathlib import Path
import sys

project_dir = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(project_dir) not in sys.path:
    sys.path.append(str(project_dir))

from src.data.backblaze_dataset_builder import build_dataset, summarize_split


In [ ]:
DATA_DIR = Path(r"E:\ITMO\2 семестр\MLops\Datasets")
MODEL_NAME = "HGST HUH721212ALN604"
HORIZON_DAYS = 7
TEST_SIZE = 0.2
NEGATIVE_RATIO = 5

OUTPUT_DIR = project_dir / "data" / "raw"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_DIR


## Сборка датасета

Запускается pipeline. По умолчанию тот же `model`, который использовался в старом ноутбуке.

Можно заменить `MODEL_NAME` на другой.


In [ ]:
full_df, train_df, test_df = build_dataset(
    data_dir=DATA_DIR,
    model_name=MODEL_NAME,
    horizon_days=HORIZON_DAYS,
    test_size=TEST_SIZE,
    negative_ratio=NEGATIVE_RATIO,
    random_state=42,
)

summary = summarize_split(full_df, train_df, test_df)
summary


In [ ]:
full_to_save = full_df.drop(columns=["serial_number"])
train_to_save = train_df.drop(columns=["serial_number"])
test_to_save = test_df.drop(columns=["serial_number"])

full_to_save.to_parquet(OUTPUT_DIR / "full_dataset.parquet", index=False)
train_to_save.to_parquet(OUTPUT_DIR / "train.parquet", index=False)
test_to_save.to_parquet(OUTPUT_DIR / "test.parquet", index=False)

sorted(path.name for path in OUTPUT_DIR.glob("*.parquet"))


In [ ]:
train_df.head(3)


In [ ]:
test_df.head(3)


## Датасет корректнее предыдущего так как:

- `label` строится по будущему отказу в горизонте 7 дней, а не по `failure` текущего дня.
- Строки дня отказа исключаются, поэтому модель учится предсказывать событие заранее.
- Все статистики и `delta` считаются только по прошлым значениям внутри одного диска.
- В train и test нет пересечения по `serial_number`, проверка стала честнее.
- Downsampling делается только в train. Тест не балансируем специально, чтобы метрики были ближе к реальной задаче.
- Убираем правое цензурирование на конце периода, чтобы случайно не размечать сомнительные строки как отрицательные.


## Ошибки исправлены:

- Убрана утечка через split по строкам: теперь разбиение идет по дискам.
- Убрана зависимость от будущей информации при формировании train/test.
- Не используются идентификаторы инфраструктуры и сам `serial_number` как признаки.
- Не используется информация после момента предсказания для rolling и `delta`.
- Балансировка сделана аккуратнее: она влияет только на обучение, но не искажает тест.
